### Burrows-Wheeler Transform:
#### Griffin Kramer
The Burrows-Wheeler algorithm takes a string and generates all possible rotations of said string, and can be represented as a matrix. The Burrows-Wheeler transform is the final column of the matrix. From the transform, first-last matching can be used to quickly count the instances of a substring pattern within the larger string. The steps for generating the Burrows-Wheeler transform are as follows:

1. Create all rotations.
2. Sort rotations alphabetically.
3. Take last column.

In [1]:
#rotate input string by 1 character
def rotate(string):
    start = string[0] #first character in string
    output = "" #output string that we will concatenate characters to 
    for i in range(1, len(string)): #iterate through string from second character (first index) to end
        output += string[i] #add current character to string
    output += start #add the first character to the end
    return output

#create the Burrows-Wheeler transform (last column of the sorted rotation matrix)
def BurrowsWheeler(string):
    string += "$" #add $ to note the end of the string (not necessary for this assignment but good practice)
    matrix = [string] #create burrows-wheeler matrix
    for i in range (len(string)-1): #iterate through string 
        matrix.append(rotate(matrix[i])) #append each rotation
    matrix.sort() #sort matrix
    
    #make transform (last column)
    transform = ""
    for i in range(len(matrix)):
        transform += matrix[i][len(string)-1] #([row] [last column])
        
    return transform

### Inverse Burrows-Wheeler
The purpose of the inverse of the Burrows-Wheeler transform lies in first-last matching. By sorting the transform, we return to the first column of the Burrows-Wheeler matrix (since it is sorted alphabetically). We could then recreate the entire matrix from solely the transform. The steps to reacquire the matrix and the sorted transform are as follows:
1. Sort transform.
2. Append sorted transform to transform to create matrix.
3. Sort matrix.
4. Append matrix to transform.
5. Repeat 3 and 4 until length each row is the length of the transform.

In [8]:
def InverseBurrowsWheeler(transform):
    #sorting the transform gives the first column of the Burrows-Wheeler Matrix
    transform_list = list(transform)
    transform_list.sort()
    sorted_transform = "".join(transform_list)
    
    #recreate original matrix
    matrix = []
    for i in range(len(transform)):
        matrix.append(sorted_transform[i])
        
    for i in range (len(transform)-1):
        for j in range (len(matrix)):
            matrix[j] =  transform[j] + matrix[j]
        matrix.sort()
    
    return(sorted_transform, matrix)

### Implementation of the Wavelet Tree Class 

Wavelet trees are not inherently necessary for first-last matching, but prevent the need to store the ranks of each character on the inverse of the transform. I chose to implement wavelet trees in first last matching to practice with python classes and recursive functions. 

To make the optimal wavelet tree, child nodes should have approximately an even number characters. If the algorithm was only intended to be used on DNA and we know some basic facts about the sequence (for example, in Human DNA we expect lower %GC content overall, but higher %GC within genes) we could predetermine the structure of the wavelet tree. To make this more broadly useful, I chose to write this class to allow for creation of the optimal split of characters within a string. This allows for the creation of uneven wavelet trees. Furthermore, it allows the first-last matching to work on any string, not just those containing only a,g,t,c. Below is an example of how the class would handle creating a wavelet tree for string "aaaaapple".

<img src="wavelet_tree_diagram.png" alt="wavelet tree diagram" width="400">

Inside Wtree class I included a rank function that is later applied to first last matching.

In [3]:
#Wavelet tree class. I chose to make this a class for organization and reusability purposes. 
#I also wanted to practice writing classes, since I don't have that much experience with OOP in python. 
class Wtree:
    def __init__(self, seq):
        self.seq = seq
        self.dict_tree = []
        
    '''
    Recursive function that takes a dictionary with counts of each  character in a string and finds 
    the best way to split it. This function is recursive because it starts with the parent, and then 
    is called again on both children (with them becoming the parents) until the split is complete.
    '''
    def even_split(self, dictionary):
        # if we aren't at the leaves of the wavelet tree
        if (len(dictionary) > 1):
            #sort the character dictionary from most present to least present in the string
            sorted_dict = sorted(dictionary.items(), key=lambda item: item[1], reverse=True)
            #dictionaries for children 
            dict1 = {}
            dict2 = {}
            #sums to best ensure even split
            sum1 = 0 
            sum2 = 0 

            #for each character and it's count in the dictionary chose which child to add it to while
            #balancing the sizes of both children using the sums.
            for key, value in sorted_dict:
                if (sum1 < sum2):
                    dict1[key] = value
                    sum1 += value
                else:
                    dict2[key] = value
                    sum2 += value

            #add children dictionaries to the dictionary tree (essentially a dataframe of dictionaries, 
            #but stored in a multidimensional list). 
            self.dict_tree.append([dict1, dictionary])
            self.dict_tree.append([dict2, dictionary])
            #recursively call the function on the children (recursion)
            self.even_split(dict1)
            self.even_split(dict2)     
     
    #find the generation of each dictionary to allow for future conversion of dictionary tree into wavelet tree.
    def find_generation(self, parent, c):
        #if original parent found (top of the tree), return c (count). 
        if parent == 0:
            return c
        #not first generation (top of the tree).
        else:
            #iterate through tree.
            for i in range(len(self.dict_tree)):
                #find parent.
                if parent == self.dict_tree[i][0]:
                    #when parent found, increase the counter.
                    c += 1
                    #then find the parents parent until at top of tree. 
                    return self.find_generation(self.dict_tree[i][1], c)
    
    #find the appropriate children so that string can be converted into 0s and 1s
    def find_children(self, string, g):
        #children output list
        children = []
        #find parent
        for i in range(len(self.dict_tree)):
            #if correct generation g
            if (self.dict_tree[i][2] == g):
                #if all characters in dictionary, correct parent found
                if all(character in self.dict_tree[i][0] for character in string):
                    parent = self.dict_tree[i][0]
                    
        #use parent to find children
        for i in range(len(self.dict_tree)):
            #if parent found
            if self.dict_tree[i][1] == parent:
                 #add child to list of children
                children.append(self.dict_tree[i][0])
        return (children[0], children[1])
            
    #converts the dictionary tree to wavelet tree, 
    def convert_to_binary(self, string, g):
        
        #if at tree leaf (only 1 type of character left)
        if string.count(string[0]) == len(string):
            #add leaf to tree
            to_add = int(self.key[string[0]], 2)
            self.wavelet_tree[g][to_add] = string[0]
        
        #if at branch (not leaf)
        else:
            #get children
            child1, child2 = self.find_children(string, g)
            #initialize variables
            new_string_1, new_string_2, output = "", "", ""
            #convert string to 0s and 1s
            for i in range(len(string)):
                character = string[i]
                #if character in child 1, add character to string1, add 0 to output, and update key if needed
                if character in child1:
                    new_string_1 += character
                    output += '0' 
                    if len(self.key[character]) < g+1:
                        self.key[character] += '0'
                #if character in child 2, add character to string2, add 1 to output, and update key if needed
                else:
                    new_string_2 += character
                    output += '1'
                    if len(self.key[character]) < g+1:
                        self.key[character] += '1'
            
            #update wavelet tree
            if g == 0:
                to_add = 0
            else:
                to_add = int(self.key[string[0]][:-1], 2)
            #print(g, output, self.key[string[0]], to_add)
            self.wavelet_tree[g][to_add] = output
            #call function on children 
            self.convert_to_binary(new_string_1, g+1)
            self.convert_to_binary(new_string_2, g+1)                
    
    #initialization function  
    def create_tree(self):
        '''
        Create the optimal wavelet tree. To do this I want to split the string such that there are 
        approximately an even number characters in the two children of each parent. 
        '''

        my_dict = {} # initialize dictionary
        
        #for loop to count the number of each character in the sequence (string) and add to dictionary
        for character in self.seq:
            if character in my_dict:
                my_dict[character] += 1
            else:
                my_dict[character] = 1 
        
        # add top node of tree to dictionary
        self.dict_tree.append([my_dict, 0])
        # create all the other branches and leaves
        self.even_split(my_dict)
        
        # add which generation (depth) to dictionary tree to make it possible to link children
        # to the correct dictionary during conversion to wavelet tree
        max_gen = 0
        for i in range(len(self.dict_tree)):

            self.dict_tree[i].append(self.find_generation(self.dict_tree[i][1], 0))
            if self.dict_tree[i][2] > max_gen:
                max_gen = self.dict_tree[i][2]
        
        #set up wavelet tree structure using max generation 
        self.wavelet_tree = []
        for i in range(max_gen+1):
            self.wavelet_tree.append([])
            for j in range (2**i):
                self.wavelet_tree[i].append("")
                
        '''
        Create key and convert dictionary tree to wavelet tree
        '''
        self.key = {char: "" for char in self.seq} 
        self.convert_to_binary(self.seq, 0)
    
    #wavelet tree rank function
    #Wavelet Tree rank_c(i) gives the number of times query character c[i] occured left in the string
    def rank(self, character, i):
        my_key = self.key[character] #get key for desired character
        count = 0 #final count
        pcount = i #temp parent count
        
        #for each character in key 
        for j in range(len(my_key)):
            
            if j == 0:
                current_key = 0
            else:
                current_key = int(my_key[:j], 2)
            
            #get current node of tree
            current_node = self.wavelet_tree[j][current_key]
            #get binary code from key at current node
            current_binary = int(my_key[j])
            barrier = pcount #set a barrier for rank based on i/pcount
            pcount = 0 #reset pcount to 0 
            
            #iterate through node
            for k in range (len(current_node[:barrier])):
                
                #if not final node
                if j != len(my_key)-1:
                    #count
                    if int(current_node[k]) == current_binary:
                        pcount += 1
                #if final node
                else:
                    if int(current_node[k]) == current_binary:
                        count += 1
                        
                #print(current_node, current_binary, pcount, count)
        
        return count

### Implementing First-Last Matching search using FM Index
To implement first last matching, I chose to convert the transform to a wavelet tree, allowing for the use of the rank function on it. First-Last matching works as follows: 

For each character in pattern starting with the last character and moving to the first, 
1. find the start and end indices of that character (c) from the pattern in the sorted transform within the restricted region (at the start no region restriction). 
2. Take $rank_c(start)$ and $rank_c(end)$ to create a restricted region, stored below as rank_start and rank_end. This region essentially gets rid of substring length sequences as the stop matching the pattern. 
3. Repeat 1 and 2 with the new restrictions and next character in the pattern until end of pattern is reached. At this point rank_end - rank_start is the number of times the pattern is present in the string.


In [4]:
def FMindex_count(sorted_transform, transform, pattern):
    #convert transform to wavelet tree. 
    wvtree = Wtree(transform)
    wvtree.create_tree()
    
    #rank start and end represent the indices where the pattern starts in the sorted transform. 
    #Since this is the beginning of the search, rank start = 0 and rank end is the end of the transform. 
    rank_start = 0 
    rank_end = len(sorted_transform)
    
    # loop from end of pattern to front
    for i in range (len(pattern)-1, 0, -1):
        
        #initialize start, end, last match, and count
        start, end, last_match = None, None, None
        count = 0 
        
        #iterate through sorted transform
        for j in range(len(sorted_transform)):
            
            #if character in sorted transform = current pattern character
            if sorted_transform[j] == pattern[i]:
                #set start at the rank_start-th occurrence
                if (start is None and count == rank_start):
                    start = j
                count += 1
                last_match = j 
                #when we reach rank_end-th occurrence, set end = j+1 (exclusive)
                if count == rank_end:
                    end = j + 1
            
        #if no matches found
        if start is None:
            return 0
        
        #if we never reached rank_end occurrences, end goes to end of array
        if end is None:
            end = last_match + 1
        
        #take the start and end from the sorted transform
        #get the rank(c, i) for both the start and end in the wavelet tree of the transform
        rank_start = wvtree.rank(pattern[i-1], start)
        rank_end = wvtree.rank(pattern[i-1], end)
    
    #return the final count 
    return rank_end - rank_start

### Testing the algorithm. 
To demonstrate that everything is working I wrote several test cases testing different factors.

Limitations:
1 character patterns break the code, putting it into an infinite loop, but there isn't a realistic use case for this algorithm when trying to count the number of times a character occurs in a string.


In [17]:
# Test Cases

# 2 character pattern
transform = BurrowsWheeler("aatggtgatg")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 3")
print(FMindex_count(sortedTransform, transform, 'tg'))

#3 character pattern 
transform = BurrowsWheeler("aatggtgatg")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 2")
print(FMindex_count(sortedTransform, transform, 'atg'))

#4 character pattern
transform = BurrowsWheeler("aatggtgatg")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 1")
print(FMindex_count(sortedTransform, transform, 'aatg'))

#Pattern not fount in string
transform = BurrowsWheeler("aatggtgatg")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 0")
print(FMindex_count(sortedTransform, transform, 'aaa'))

#Pattern overlaps within string
transform = BurrowsWheeler("cgcatcgcattttcgcaacgca")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 2")
print(FMindex_count(sortedTransform, transform, "ttt"))

#Non-DNA string
transform = BurrowsWheeler("mississippi")
sortedTransform, matrix = InverseBurrowsWheeler(transform)
print(transform, sortedTransform)
print("Expected output: 2")
print(FMindex_count(sortedTransform, transform, "issi"))

g$gatttgaga $aaaggggttt
Expected output: 3
3
g$gatttgaga $aaaggggttt
Expected output: 2
2
g$gatttgaga $aaaggggttt
Expected output: 1
1
g$gatttgaga $aaaggggttt
Expected output: 0
0
accaccggggat$tcccctatta $aaaaaccccccccggggttttt
Expected output: 2
2
ipssm$pissii $iiiimppssss
Expected output: 2
2


### Final Implementation

In [40]:
file = open(r"ref-for-bwt-search.fa")
#this gets the sequence code.
for line in file:
    if line[0] != '>':
        sequence = line

transform = BurrowsWheeler(sequence)
sortedTransform, matrix = InverseBurrowsWheeler(transform)

print("Number of times AGATA is in sequence:")
print(FMindex_count(sortedTransform, transform, "AGATA"))

print("Number of times TATCT is in sequence:")
print(FMindex_count(sortedTransform, transform, "TATCT"))


Number of times AGATA is in sequence:
5
Number of times TATCT is in sequence:
3


### Other implementations
Due to its distinct structure and stereochemical restrictions pertaining to phi and psi angles, proline residues in amino acid primary structure are often an indication of sharp turns in protein folding. IGHG1 contains $\beta$-sheets connected by loop regions resulting in several of these sharp turns. We can count potential sharp turn sites by looking at proline-proline, glycine-proline, and proline-glycine residues in the primary structure. 

In [53]:
#sequence for IGHG1 from Uniprot
IGHG1 = "ASTKGPSVFPLAPCSRSTSESTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTKTYTCNVDHKPSNTKVDKRVESKYGPPCPSCPAPEFLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSQEDPEVQFNWYVDGVEVHNAKTKPREEQFNSTYRVVSVLTVLHQDWLNGKEYKCKVSNKGLPSSIEKTISKAKGQPREPQVYTLPPSQEEMTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSRLTVDKSRWQEGNVFSCSVMHEALHNHYTQKSLSLSLELQLEESCAEAQDGELDGLWTTITIFITLFLLSVCYSATVTFFKVKWIFSSVVDLKQTIVPDYRNMIRQGA"

transform = BurrowsWheeler(IGHG1)
sortedTransform, matrix = InverseBurrowsWheeler(transform)

print("Number of times PP is in sequence:")
print(FMindex_count(sortedTransform, transform, "PP"))
print("Number of times PG is in sequence:")
print(FMindex_count(sortedTransform, transform, "PG"))
print("Number of times GP is in sequence:")
print(FMindex_count(sortedTransform, transform, "GP"))

Number of times PP is in sequence:
4
Number of times PG is in sequence:
0
Number of times GP is in sequence:
3
